# Dzien 2 / Krok 4: FE gotowe na produkcje

**Input:** `checkpoints/01_merged.parquet`
**Output:** `checkpoints/04_pipeline.pkl`

Ten sam Feature Engineering co w Kroku 2, przepisany na wzorzec produkcyjny:
- `df.pipe`: kazdy krok FE to czysta funkcja
- `FeatureEngineeringTransformer`: owijka sklearn fit/transform
- `VolumeDecompTransformer`: dekompozycja jako transformer
- sklearn `Pipeline`: surowe dane -> predykcja w jednym obiekcie

Serwis produkcyjny: `pipe.predict_proba(df_raw_order)[0, 1]`

Analogia do backendu: Pipeline = middleware chain. Kazdy transformer to jeden middleware.


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import joblib

df = pd.read_parquet("checkpoints/01_merged.parquet")
print(f"Zaladowano: {df.shape}")


## Feature Engineering przez df.pipe

Kazda funkcja: przyjmuje DataFrame, zwraca DataFrame, nie modyfikuje oryginalu.

Analogia: pure functions — bez side effects, latwe w unit testach, skladalne w dowolnej kolejnosci.


In [ ]:
# TODO: Zaimplementuj cztery czyste funkcje FE
# Kazda: df: pd.DataFrame -> pd.DataFrame (zaczynaj od feat = df.copy())

def _add_delivery_features(df):
    # TODO: estimated_delivery_days (clip 0-120)
    # TODO: approval_time_hours (clip 0-48, fillna 24)
    pass

def _add_time_features(df):
    # TODO: purchase_hour, purchase_dow, purchase_month, is_weekend
    pass

def _add_price_features(df):
    # TODO: price_log (np.log1p), freight_ratio
    pass

def _add_payment_features(df):
    # TODO: payment_installments (fillna 0), is_installment
    pass

def feature_engineering(df):
    # todo: df.pipe(callback_1).pipe(callback_2)...
    ...
    
# Test izolowany - sprawdz ze funkcje dzialaja
sample = df.head(3)
result = feature_engineering(sample)
print(f"Wejscie: {sample.shape[1]} kolumn  ->  Wyjscie: {result.shape[1]} kolumn")
expected_new = ["estimated_delivery_days", "approval_time_hours",
                "purchase_hour", "purchase_dow", "purchase_month", "is_weekend",
                "price_log", "freight_ratio", "payment_installments", "is_installment"]
missing = [c for c in expected_new if c not in result.columns]
print(f"Brakujace kolumny: {missing if missing else 'brak - OK'}")


## FeatureEngineeringTransformer: df.pipe jako sklearn transformer

Owijamy lancuch `df.pipe` w sklearn transformer.

`fit()` zapamietuje to co jest stanowe: enkodery kategorii i mediane wagi.
Funkcje `_add_*` sa bezstanowe — nie wymagaja fit.

**Kluczowy wzorzec produkcyjny:** LabelEncoder zawiera kategorie `"unknown"` —
nieznane wartosci w produkcji nie rzuca bledow, sa bezpiecznie mapowane.


In [ ]:
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    """
    fit:       zapamietuje LabelEncoder per kolumna + mediane wagi
    transform: aplikuje feature_engineering() + enkoduje kategorie
    """

    CAT_COLS = ["payment_type", "product_category_name_english",
                "customer_state", "seller_state"]

    def fit(self, X, y=None):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        df_base = feature_engineering(df)

        self.label_encoders_ = {}
        for col in self.CAT_COLS:
            le = LabelEncoder()
            # TODO: Naucz LabelEncoder
            # Wazne: dolacz "unknown" do listy wartosci przed fit()
            # -> transform() bedzie mogl mapowac nieznane kategorie na "unknown"
            values = ___
            le.fit(values)
            self.label_encoders_[col] = le

        # TODO: Zapamietaj mediane wagi (do imputacji braków w transform)
        self.weight_median_ = ___
        return self

    def transform(self, X):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        feat = feature_engineering(df)

        for col, le in self.label_encoders_.items():
            vals = feat[col].fillna("unknown").astype(str)
            # TODO: Mapuj wartosci spoza treningu na "unknown"
            known = set(le.classes_)
            vals  = ___
            feat[col + "_enc"] = le.transform(vals)

        # TODO: Uzupelnij braki - weight_g -> self.weight_median_, photos_qty -> 1
        feat["product_weight_g"]   = ___
        feat["product_photos_qty"] = ___
        return feat


## VolumeDecompTransformer: dekompozycja jako sklearn transformer

`fit(X_train)`: oblicza lookup z danych treningowych
`transform(X)`: mapuje trend/seasonal na kazdy wiersz przez rok-miesiac

**Uwaga:** `seasonal_decompose` wymaga >= 24 miesiecy (`2 * period`).
Po 80/20 splicie train moze miec mniej - dlatego mamy parametr `decomp_lookup`
do przekazania lookup obliczonego na pelnym zbiorze.


In [ ]:
class VolumeDecompTransformer(BaseEstimator, TransformerMixin):
    """
    decomp_lookup: jezeli podany, fit() uzywa gotowego lookup
                   (dla krotkiego zbioru treningowego < 24 miesiecy)
    """

    def __init__(self, ts_col="order_purchase_timestamp", period=12,
                 decomp_lookup=None):
        self.ts_col        = ts_col
        self.period        = period
        self.decomp_lookup = decomp_lookup

    def fit(self, X, y=None):
        if self.decomp_lookup is not None:
            self.decomp_lookup_ = self.decomp_lookup
            return self

        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)

        # TODO: Oblicz miesięczny wolumen i wykonaj dekompozycje
        monthly = ___

        if len(monthly) < 2 * self.period:
            raise ValueError(
                f"Za malo miesiecy: {len(monthly)} (wymagane >= {2 * self.period}). "
                f"Uzyj decomp_lookup= z lookup obliczonym na pelnym datasecie."
            )

        result = ___  # TODO: seasonal_decompose

        # TODO: Zbuduj self.decomp_lookup_ DataFrame z indeksem Period("M")
        self.decomp_lookup_ = ___
        self.decomp_lookup_.index = self.decomp_lookup_.index.to_period("M")
        return self

    def transform(self, X):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        df = df.copy()
        # TODO: periods = ...to_period("M"), zmapuj trend i seasonal
        periods = ___
        df["volume_trend"]    = ___
        df["volume_seasonal"] = ___
        return df


In [ ]:
TARGET = "is_bad_experience"

y      = df[TARGET]
X_raw  = df.drop(columns=[TARGET])

X_raw_train, X_raw_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# Fitujemy na pelnym X_raw - gwarantuje 24 miesiace dla dekompozycji
vdt_ref = VolumeDecompTransformer()
vdt_ref.fit(X_raw)

print(f"Train: {len(X_raw_train):,}  |  Test: {len(X_raw_test):,}")
print(f"Lookup: {vdt_ref.decomp_lookup_.index.min()} - {vdt_ref.decomp_lookup_.index.max()}")


## Pelny Pipeline

```
X_raw -> FeatureEngineeringTransformer -> VolumeDecompTransformer
      -> FeatureSelector -> StandardScaler -> XGBClassifier
```

Jeden obiekt zastepuje caly preprocessing kod w serwisie.


In [ ]:
FEATURE_COLS = [
    "estimated_delivery_days", "approval_time_hours",
    "purchase_hour", "purchase_dow", "purchase_month", "is_weekend",
    "volume_trend", "volume_seasonal",
    "price_log", "freight_ratio",
    "items_count", "payment_installments", "is_installment",
    "product_weight_g", "product_photos_qty",
    "payment_type_enc", "product_category_name_english_enc",
    "customer_state_enc", "seller_state_enc",
]

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, cols): self.cols = cols
    def fit(self, X, y=None): return self
    def transform(self, X):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        return df[self.cols]

scale_pos_weight = (1 - y_train.mean()) / y_train.mean()

# TODO: Zbuduj Pipeline z krokami: FeatureEngineeringTransformer -> VolumeDecompTransformer ->  FeatureSelector, StandardScaler 
# ("fe",       FeatureEngineeringTransformer()),
# ("decomp",   VolumeDecompTransformer(decomp_lookup=vdt_ref.decomp_lookup_)),
# ("selector", FeatureSelector(FEATURE_COLS)),
# ("scaler",   StandardScaler()),
# ("model",    xgb.XGBClassifier(objective="binary:logistic", n_estimators=300,
#                                max_depth=5, learning_rate=0.05,
#                                scale_pos_weight=float(scale_pos_weight),
#                                eval_metric="auc", random_state=42, n_jobs=-1))
pipe = Pipeline([
    # TODO
])

pipe.fit(X_raw_train, y_train)
auc = roc_auc_score(y_test, pipe.predict_proba(X_raw_test)[:, 1])
print(f"Pipeline AUC: {auc:.4f}")
print(f"Kroki: {[name for name, _ in pipe.steps]}")


In [ ]:
# Inferens na jednym przykladzie
single_order = X_raw_test.iloc[[0]]
proba = pipe.predict_proba(single_order)[0, 1]
decision = "RYZYKO - wyslij voucher" if proba > 0.5 else "OK"
print(f"Prawdopodobienstwo zlej recenzji: {proba:.3f}")
print(f"Decyzja (prog=0.5): {decision}")
print()
print("W serwisie produkcyjnym: pipe = joblib.load(...), pipe.predict_proba(df_new)[0,1]")


## Zapis Pipeline

In [ ]:
import os
pipeline_path = "checkpoints/04_pipeline.pkl"
joblib.dump(pipe, pipeline_path)
print(f"Zapisano: {pipeline_path}")
print(f"Rozmiar pliku: {os.path.getsize(pipeline_path) / 1024:.0f} KB")
